# GOAL: Pixel-level masks on satelite pictures of houses.

- Use the pixel-level results from the SAM model (lots of overlapping predictions) by checking which ones "agree" with the satellite-building-segmentation labels.

In [ ]:
!pip install huggingface_hub datasets segment-anything

**Load satellite dataset**

In [ ]:
from datasets import load_dataset
# From https://huggingface.co/datasets/keremberke/satellite-building-segmentation?row=0
ds = load_dataset("keremberke/satellite-building-segmentation", name="full")
example = ds['train'][0]

In [ ]:
display(example["image"])
example["objects"]['bbox']
example.keys()

In [ ]:
example["objects"]['bbox']

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

image = example["image"].copy()  # Create a copy to avoid modifying the original

fig, ax = plt.subplots(1)
ax.imshow(image)

for bbox in example["objects"]["bbox"]:
  x_min, y_min, width, height = bbox
  rect = patches.Rectangle((x_min, y_min), width, height, linewidth=1, edgecolor='r', facecolor='none')
  ax.add_patch(rect)

plt.show()

In [ ]:
def make_mask(labelled_bbox, image):
  x_min_ones, y_min_ones, width_ones, height_ones = labelled_bbox
  x_min_ones, y_min_ones, width_ones, height_ones = int(x_min_ones), int(y_min_ones), int(width_ones), int(height_ones)
  mask_instance = np.zeros((image.width,image.height))

  last_x = x_min_ones+width_ones
  last_y = y_min_ones+height_ones
  mask_instance[x_min_ones:last_x, y_min_ones:last_y] = np.ones((int(width_ones),int(height_ones)))
  return mask_instance.T

labelled_bbox = example["objects"]["bbox"][0]
mask_instance = make_mask(labelled_bbox, image)
plt.imshow(mask_instance, cmap='gray')
plt.show()

In [ ]:
# From: https://github.com/facebookresearch/segment-anything/blob/main/notebooks/automatic_mask_generator_example.ipynb
using_colab=True
if using_colab:
    import torch
    import torchvision
    print("PyTorch version:", torch.__version__)
    print("Torchvision version:", torchvision.__version__)
    print("CUDA is available:", torch.cuda.is_available())
    import sys
    !{sys.executable} -m pip install opencv-python matplotlib
    !{sys.executable} -m pip install 'git+https://github.com/facebookresearch/segment-anything.git'

    !mkdir images
    !wget -P images https://raw.githubusercontent.com/facebookresearch/segment-anything/main/notebooks/images/dog.jpg

    !wget https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
import cv2

def show_anns(anns):
    if len(anns) == 0:
        return
    sorted_anns = sorted(anns, key=(lambda x: x['area']), reverse=True)
    ax = plt.gca()
    ax.set_autoscale_on(False)
    img = np.ones((sorted_anns[0]['segmentation'].shape[0], sorted_anns[0]['segmentation'].shape[1], 4))
    img[:,:,3] = 0
    for ann in sorted_anns:
        m = ann['segmentation']
        color_mask = np.concatenate([np.random.random(3), [0.35]])
        img[m] = color_mask
    ax.imshow(img)

In [ ]:
import sys
sys.path.append("..")
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator, SamPredictor

sam_checkpoint = "sam_vit_h_4b8939.pth"
model_type = "vit_h"

device = "cuda"

sam = sam_model_registry[model_type](checkpoint=sam_checkpoint)
sam.to(device=device)

mask_generator = SamAutomaticMaskGenerator(sam)

In [ ]:
masks = mask_generator.generate(np.array(example["image"]))

In [ ]:
print(len(masks))
print(masks[0].keys())

In [ ]:
binary_array = masks[10]['segmentation'].astype(int)
plt.imshow(binary_array, cmap='gray')
plt.show()

In [ ]:
for m in masks:
  print(m['bbox'])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

image2 = example["image"].copy()  # Create a copy to avoid modifying the original

fig, ax = plt.subplots(1)
ax.imshow(image)

for m in masks:
  bbox = m['bbox']
  x_min, y_min, width, height = bbox
  rect = patches.Rectangle((x_min, y_min), width, height, linewidth=1, edgecolor='r', facecolor='none')
  ax.add_patch(rect)

plt.show()

In [ ]:
plt.figure(figsize=(10,10))
plt.imshow(image)
show_anns(masks)
plt.axis('off')
plt.show()

In [ ]:
from scipy.spatial import distance

for sam_box in masks:
  sam_seg = sam_box['segmentation'].astype(int)
  for label_box in example["objects"]["bbox"]:
    label_seg = make_mask(label_box, image)
    iou = np.sum(np.logical_and(sam_seg, label_seg)) / np.sum(np.logical_or(sam_seg, label_seg))
    if iou>0.3:
      print(iou)
      # Create a figure with 1 row and 2 columns
      fig, axes = plt.subplots(1, 2, figsize=(10, 5))

      # Display the first image in the first subplot
      axes[0].imshow(label_seg, cmap='gray')
      axes[0].set_title('Labelled Segmentation')
      axes[0].axis('off')  # Hide axes for better visualization

      # Display the second image in the second subplot
      axes[1].imshow(sam_seg, cmap='gray')
      axes[1].set_title('SAM Segmentation')
      axes[1].axis('off')  # Hide axes for better visualization

      # Adjust spacing between subplots
      plt.tight_layout()
      plt.show()